# Real-World Retail Dataset Cleaning (1M+ Rows)

**Dataset:** UCI Online Retail II (1,067,371 transactions; UK online retailer; Dec 2009–Dec 2011).  
**Source:** UCI Machine Learning Repository, dataset DOI `10.24432/C5CG6D` (CC BY 4.0).

This notebook loads the real transaction dataset, captures a before-cleaning profile, standardizes data types and text, handles missing values and duplicates, flags/removes invalid transaction rows, treats extreme values transparently, engineers date and sales features, and exports `clean_dataset.csv`.

> **Important business caveat:** the source has quantity and unit selling price but no product cost. True profit margin cannot be calculated from the original fields. The notebook calculates sales revenue and only calculates profit margin if a cost field is available. It does not invent costs.


## 1. Install and import libraries

In [ ]:
# Run once if required:
# %pip install ucimlrepo pandas numpy openpyxl

import pandas as pd
import numpy as np
from pathlib import Path
import re
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 2. Load the raw dataset

In [ ]:
# Preferred: download the official UCI dataset through ucimlrepo.
# If this download method is unavailable, download Online Retail II from the UCI page
# and place online_retail_II.xlsx in this notebook's folder.

DATA_FILE = Path("online_retail_II.xlsx")

if DATA_FILE.exists():
    raw = pd.read_excel(DATA_FILE, sheet_name=0)
else:
    try:
        from ucimlrepo import fetch_ucirepo
        dataset = fetch_ucirepo(id=502)
        raw = dataset.data.features.copy()
    except Exception as exc:
        raise RuntimeError(
            "Could not download the UCI dataset automatically. Download "
            "'online_retail_II.xlsx' from the official UCI Online Retail II page "
            "and place it beside this notebook, then rerun this cell."
        ) from exc

print("Raw shape:", raw.shape)
display(raw.head())
print("\nRaw data types:")
display(raw.dtypes.to_frame("dtype"))


## 3. Before-cleaning data profile

In [ ]:
before_shape = raw.shape
before_missing = raw.isna().sum().sort_values(ascending=False)
before_duplicates = int(raw.duplicated().sum())

before_profile = pd.DataFrame({
    "dtype_before": raw.dtypes.astype(str),
    "missing_count": raw.isna().sum(),
    "missing_pct": (raw.isna().mean() * 100).round(2),
    "unique_values": raw.nunique(dropna=True)
})
print(f"Rows before cleaning: {before_shape[0]:,}")
print(f"Columns before cleaning: {before_shape[1]}")
print(f"Exact duplicate rows: {before_duplicates:,}")
display(before_profile)


## 4. Standardize column names, text and data types

In [ ]:
df = raw.copy()

# Normalize column labels to snake_case.
def clean_column_name(name):
    name = str(name).strip()
    name = re.sub(r"[^0-9a-zA-Z]+", "_", name)
    return name.strip("_").lower()

df.columns = [clean_column_name(c) for c in df.columns]

# Normalize common source column names.
rename_map = {
    "invoiceno": "invoice_no",
    "stockcode": "stock_code",
    "description": "description",
    "quantity": "quantity",
    "invoicedate": "invoice_date",
    "unitprice": "unit_price",
    "customerid": "customer_id",
    "country": "country",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

# IDs are identifiers, not numeric measures.
for col in ["invoice_no", "stock_code", "customer_id"]:
    if col in df:
        df[col] = df[col].astype("string").str.strip()

# Standardize text fields.
for col in ["description", "country"]:
    if col in df:
        df[col] = df[col].astype("string").str.strip()
        df[col] = df[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

# Convert numeric values safely; invalid text becomes missing and is handled below.
for col in ["quantity", "unit_price"]:
    if col in df:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "invoice_date" in df:
    df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce", dayfirst=True)

if "customer_id" in df:
    # IDs remain strings; blank and textual nan become missing.
    df["customer_id"] = df["customer_id"].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

print(df.dtypes)
display(df.head())


## 5. Handle duplicates and missing values

In [ ]:
# Remove exact duplicate transaction lines.
duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

# Missing customer IDs are retained as 'Unknown' because anonymous purchases are valid
# transactions in this source. Missing descriptions are filled from product code where possible.
if "customer_id" in df:
    df["customer_id"] = df["customer_id"].fillna("Unknown")

if {"description", "stock_code"}.issubset(df.columns):
    df["description"] = df["description"].fillna(
        "Unknown product (" + df["stock_code"].astype("string") + ")"
    )

# Country is a categorical descriptor; use an explicit label rather than dropping the sale.
if "country" in df:
    df["country"] = df["country"].fillna("Unknown")

# Critical transaction fields cannot be safely imputed: remove rows without invoice/date/quantity/price.
critical = [c for c in ["invoice_no", "invoice_date", "quantity", "unit_price"] if c in df.columns]
rows_before_critical = len(df)
df = df.dropna(subset=critical).copy()
critical_rows_removed = rows_before_critical - len(df)

print("Exact duplicates removed:", duplicate_count)
print("Rows removed for missing critical fields:", critical_rows_removed)
print("Remaining missing values:")
display(df.isna().sum().to_frame("missing_count"))


## 6. Handle invalid transactions and outliers

In [ ]:
# In this dataset, negative quantities and invoices beginning with C represent returns/cancellations.
# Keep them as valid business events, but classify them explicitly.
df["transaction_type"] = np.where(
    df["invoice_no"].str.upper().str.startswith("C") | (df["quantity"] < 0),
    "Return_or_Cancellation",
    "Sale"
)

# Unit prices at or below zero are not valid sales prices for revenue calculations.
invalid_price_rows = int((df["unit_price"] <= 0).sum())
df = df.loc[df["unit_price"] > 0].copy()

# Quantity zero has no sale/return volume; exclude from the cleaned transaction fact table.
zero_quantity_rows = int((df["quantity"] == 0).sum())
df = df.loc[df["quantity"] != 0].copy()

# Outlier detection: IQR flags extreme positive quantities and prices.
# Do not automatically delete legitimate wholesale orders. Preserve rows and add flags.
def iqr_bounds(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

q_low, q_high = iqr_bounds(df.loc[df["quantity"] > 0, "quantity"])
p_low, p_high = iqr_bounds(df["unit_price"])

df["quantity_outlier_flag"] = (df["quantity"] > 0) & (
    (df["quantity"] < q_low) | (df["quantity"] > q_high)
)
df["unit_price_outlier_flag"] = (
    (df["unit_price"] < p_low) | (df["unit_price"] > p_high)
)

print("Rows removed for non-positive unit price:", invalid_price_rows)
print("Rows removed for zero quantity:", zero_quantity_rows)
print("Quantity outlier flags:", int(df["quantity_outlier_flag"].sum()))
print("Unit price outlier flags:", int(df["unit_price_outlier_flag"].sum()))


## 7. Feature engineering

In [ ]:
df["year"] = df["invoice_date"].dt.year.astype("Int64")
df["month"] = df["invoice_date"].dt.month.astype("Int64")
df["month_name"] = df["invoice_date"].dt.month_name()
df["year_month"] = df["invoice_date"].dt.to_period("M").astype("string")
df["day_of_week"] = df["invoice_date"].dt.day_name()

# Line revenue is quantity multiplied by unit selling price.
# Negative quantities remain negative to represent returns.
df["sales_revenue"] = (df["quantity"] * df["unit_price"]).round(2)

# The source does not include cost of goods sold. Calculate profit/margin only if a cost
# column exists in a future version of the source; never fabricate cost values.
cost_candidates = ["unit_cost", "cost_price", "cost"]
available_cost = next((c for c in cost_candidates if c in df.columns), None)

if available_cost:
    df[available_cost] = pd.to_numeric(df[available_cost], errors="coerce")
    df["profit"] = df["sales_revenue"] - (df["quantity"] * df[available_cost])
    df["profit_margin_pct"] = np.where(
        df["sales_revenue"] != 0,
        (df["profit"] / df["sales_revenue"]) * 100,
        np.nan
    )
else:
    print("Profit margin not calculated: the official dataset has no cost field.")
    print("A valid margin requires cost data: (Revenue - Cost) / Revenue × 100.")


## 8. Final validation and after-cleaning profile

In [ ]:
# Enforce final types and remove any remaining exact duplicates.
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
df = df.drop_duplicates().reset_index(drop=True)

after_profile = pd.DataFrame({
    "dtype_after": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True)
})

print(f"Rows after cleaning: {len(df):,}")
print(f"Columns after cleaning: {len(df.columns)}")
print(f"Rows removed overall: {before_shape[0] - len(df):,}")
print("Remaining exact duplicates:", int(df.duplicated().sum()))
display(after_profile)
display(df.head())


## 9. Export clean dataset

In [ ]:
OUTPUT_FILE = "clean_dataset.csv"
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"Saved {OUTPUT_FILE} with {len(df):,} rows and {len(df.columns)} columns.")


## 10. Summary of cleaning decisions

- Exact duplicate rows: removed.
- Customer IDs missing: retained as `Unknown` because anonymous purchases can be legitimate.
- Missing product descriptions: filled with a transparent placeholder including the stock code.
- Missing country: filled as `Unknown`.
- Missing critical invoice/date/quantity/price fields: rows removed because safe imputation is not possible.
- Invalid numeric strings: converted to missing, then handled by critical-field rules.
- Negative quantities and cancellation invoices: retained and labeled as returns/cancellations.
- Zero quantities and non-positive unit prices: excluded from the cleaned transaction fact table.
- Extreme quantity/price values: flagged, not automatically deleted, because wholesale purchases can be legitimate.
- Date features and sales revenue: engineered.
- Profit margin: only computed if cost data exists; the official source does not supply cost.

**Dataset citation:** Chen, D. (2012). *Online Retail II* [Dataset]. UCI Machine Learning Repository. DOI: `10.24432/C5CG6D`. Licensed under CC BY 4.0.
